In [8]:
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans


In [9]:
data_dir = Path("data")
INDIR = Path("data/data_processed")
OUTDIR = Path("data/data_model")

OUTDIR.mkdir(parents=True, exist_ok=True)

In [10]:
arquivo_cluster = INDIR / "PARTICIPANTES_SIMPLIFICADO_TRATADO.csv"
arquivo_scale = INDIR / "PARTICIPANTES_SIMPLIFICADO_TRATADO_MODELO.csv"

df_clustering = pd.read_csv(arquivo_cluster, encoding='utf-8')
X_scaled = pd.read_csv(arquivo_scale, encoding='utf-8')

In [ ]:
kmeans = KMeans(n_clusters=2, n_init=1000, random_state=0)
kmeans.fit(X_scaled)

labels = kmeans.labels_
df_clustering['CLUSTER'] = labels

In [ ]:
caminho_cluster = OUTDIR / "PARTICIPANTES_CLUSTERS_2.csv"
df_clustering.to_csv(caminho_cluster, index=False)

In [13]:
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=X_scaled.columns
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

display(centroids.round(2))

,RENDA_FAMILIAR_SM_LOG,IDADE_LOG,ESTADO_CIVIL_DIVORCIADO,ESTADO_CIVIL_SOLTEIRO,ESTADO_CIVIL_UNIAO,COR_RACA_AMARELA,COR_RACA_BRANCA,COR_RACA_INDÍGENA,COR_RACA_PARDA,COR_RACA_PRETA,SEXO_F,SEXO_M,REGIAO_CENTRO_OESTE,REGIAO_NORDESTE,REGIAO_NORTE,REGIAO_SUDESTE,REGIAO_SUL
0,1.00,-0.43,0.01,0.99,0.01,0.01,0.66,0.00,0.26,0.07,0.57,0.43,0.09,0.13,0.06,0.50,0.22
1,-0.20,1.76,0.02,0.89,0.09,0.02,0.31,0.01,0.49,0.18,0.60,0.40,0.07,0.40,0.14,0.27,0.13
2,-0.59,-0.40,0.01,0.98,0.01,0.02,0.24,0.01,0.57,0.16,0.62,0.38,0.07,0.51,0.15,0.23,0.06


In [16]:
def moda_ou_na(serie):
    moda = serie.mode()
    return moda.iat[0] if not moda.empty else pd.NA

def top2_ou_na(serie):
    if serie.empty:
        return pd.NA
    return " / ".join(serie.value_counts().head(2).index.astype(str))

resumo_clusters = (
    df_clustering
    .groupby("CLUSTER")
    .agg(
        qtd_de_inscritos=("NU_INSCRICAO", "count"),
        mediana_renda_familiar_sm=("RENDA_FAMILIAR_SM", "median"),
        media_idade=("IDADE", "mean"),
        sexo_predominante=("SEXO", moda_ou_na),
        estado_civil_predominante=("ESTADO_CIVIL", top2_ou_na),
        cor_raca_predominante=("COR_RACA", top2_ou_na),
        regiao_predominante=("REGIAO", top2_ou_na),
    )
    .reset_index()
    .sort_values("CLUSTER")
    .reset_index(drop=True)
)

resumo_clusters['qtd_de_inscritos_pct'] = (
    resumo_clusters['qtd_de_inscritos'] / resumo_clusters['qtd_de_inscritos'].sum()
) * 100

resumo_clusters.columns = [col.upper() for col in resumo_clusters.columns]
display(resumo_clusters.round(2))

,CLUSTER,QTD_DE_INSCRITOS,MEDIANA_RENDA_FAMILIAR_SM,MEDIA_IDADE,SEXO_PREDOMINANTE,ESTADO_CIVIL_PREDOMINANTE,COR_RACA_PREDOMINANTE,REGIAO_PREDOMINANTE,QTD_DE_INSCRITOS_PCT
0,0,891035,2.5,18.47,F,SOLTEIRO / DIVORCIADO,BRANCA / PARDA,SUDESTE / SUL,32.47
1,1,520094,1.0,25.24,F,SOLTEIRO / UNIAO,PARDA / BRANCA,NORDESTE / SUDESTE,18.95
2,2,1333113,1.0,18.52,F,SOLTEIRO / UNIAO,PARDA / BRANCA,NORDESTE / SUDESTE,48.58
